# US 수출 데이터 SARIMA 예측 및 DB 저장 (V5)

**V5 핵심 목적: 예측치 폭발(explosion) 완전 차단**

V4까지의 문제: 로그 변환 + `(d=1, D=1)` 차분 모델은 시계열 최근 구간의
기울기(추세/가속)를 그대로 외삽하고, `exp()` 역변환이 이를 **복리로 증폭**시켜
forecast가 hist_max의 수십~수백 배, 심하면 `e+158`까지 폭발함.
→ `enforce_stationarity=True`(V4)는 '폭발 AR 근'만 막을 뿐 **추세 외삽 폭발은 못 막음**.

V4 대비 변경사항:

| # | 항목 | V4 | V5 |
|---|---|---|---|
| 1 | **Horizon-aware 봉투 클립** | 없음 | **로그공간 상/하단 봉투로 forecast 강제 제한 (폭발·붕괴 동시 차단)** |
| 2 | Spec 탐색 sanity | log 스케일 우회(1e6) | **raw(exp) 스케일에서 검증 → 폭발 spec을 '선택 단계'에서 배제** |
| 3 | 폭발 종목 처리 | forecast 전체 폐기(→실측 복사) | **bounded forecast 유지 + 구조변화 플래그** |
| 4 | 진단 메타데이터 | order/aic | **n_clipped, clip_ratio, max_fc_ratio, structural_break_flag 추가** |
| 5 | 최종 하드 가드 | hist_max×10 (폐기) | **클립 후 방어선으로 유지(거의 미발동)** |

**봉투(envelope) 원리**: 미래 h개월의 forecast 상한선을
`hist_max × (1+max_ann_growth)^(h/12)` 로 두고, 하한선도 대칭으로 설정.
→ 정상/완만한 성장 종목은 봉투 안이라 **그대로 통과**,
폭발하려는 종목만 봉투에 막혀 **bounded** 됨. `max_ann_growth=1.5`(연 150%)가 기본.

> ⚠️ **기존 DB의 폭발 행(V3/V4 run) 정리 필요**: 맨 아래 '15. 스테일 데이터 정리' 셀 참조.
> 재예측 후 ranking 쿼리는 반드시 **최신 created_at** 만 읽어야 함.

테이블 스키마는 V4와 동일 (마이그레이션 셀 그대로 사용).

## 1. 라이브러리

In [1]:
import warnings
warnings.filterwarnings("ignore")

import os
import json
import math
import datetime as dt
from datetime import datetime
from typing import Dict, Tuple, List, Optional

import numpy as np
import pandas as pd
import pymysql

from statsmodels.tsa.statespace.sarimax import SARIMAX
from pandas.tseries.offsets import MonthEnd
from tqdm.auto import tqdm

from DATA.stock_invest_function import get_db_host

## 2. DB 연결 정보

In [2]:
db_info = {
    'user':     'stox7412',
    'password': 'Apt106503!~',
    'host':     get_db_host(),
    'port':     3307,
    'database': 'investar',
    'charset':  'utf8mb4',
}

# 연결 테스트
try:
    conn = pymysql.connect(**db_info)
    with conn.cursor() as cursor:
        cursor.execute("SELECT VERSION()")
        ver = cursor.fetchone()[0]
        print(f"✓ DB 연결 성공 (MySQL {ver})")
    conn.close()
except Exception as e:
    print(f"✗ DB 연결 실패: {e}")

✓ DB 연결 성공 (MySQL 10.11.6-MariaDB)


## 3. 테이블 스키마 마이그레이션

PK가 `(hs_code, date_month_end, created_at)` 3중 키로 변경됩니다.
이렇게 두면 매일 새 run이 들어와도 누적되어 일별 예측 변화 추적 가능 + 같은 run 내 중복은 차단됩니다.

⚠️ **`force_recreate=True`로 실행하면 기존 데이터가 모두 삭제**됩니다. 
한 번만 실행하시고 그 이후엔 `force_recreate=False`로 두세요.

In [3]:
def migrate_table_schema(db_info: Dict, force_recreate: bool = False):
    """
    테이블 스키마 마이그레이션
    
    force_recreate=True : DROP + CREATE (기존 데이터 모두 삭제)
    force_recreate=False: CREATE IF NOT EXISTS (안전)
    """
    create_monthly = """
    CREATE TABLE IF NOT EXISTS us_trade_export_monthly_with_forecast (
        hs_code         VARCHAR(20) NOT NULL,
        date_month_end  DATE        NOT NULL,
        expDlr          DOUBLE      NULL,
        expDlr_forecast DOUBLE      NULL,
        is_forecast     TINYINT     NOT NULL DEFAULT 0,
        params          TEXT        NULL,
        created_at      DATETIME    NOT NULL,
        PRIMARY KEY (hs_code, date_month_end, created_at),
        INDEX idx_hs_code   (hs_code),
        INDEX idx_created_at (created_at),
        INDEX idx_is_fcst   (is_forecast)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """
    
    create_quarter = """
    CREATE TABLE IF NOT EXISTS us_trade_export_quarter_with_forecast (
        hs_code           VARCHAR(20) NOT NULL,
        date_quarter_end  DATE        NOT NULL,
        expDlr            DOUBLE      NULL,
        expDlr_forecast   DOUBLE      NULL,
        is_forecast       TINYINT     NOT NULL DEFAULT 0,
        params            TEXT        NULL,
        created_at        DATETIME    NOT NULL,
        PRIMARY KEY (hs_code, date_quarter_end, created_at),
        INDEX idx_hs_code    (hs_code),
        INDEX idx_created_at (created_at)
    ) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4
    """
    
    conn = pymysql.connect(**db_info)
    try:
        with conn.cursor() as cursor:
            if force_recreate:
                print("⚠️ 기존 테이블 DROP 중...")
                cursor.execute("DROP TABLE IF EXISTS us_trade_export_monthly_with_forecast")
                cursor.execute("DROP TABLE IF EXISTS us_trade_export_quarter_with_forecast")
                print("  → 기존 데이터 삭제 완료")
            
            cursor.execute(create_monthly)
            cursor.execute(create_quarter)
        conn.commit()
        print("✓ 테이블 스키마 준비 완료 (PK: hs_code, date_*, created_at)")
    finally:
        conn.close()


# 첫 실행 시 1회만:
# migrate_table_schema(db_info, force_recreate=True)

# 평소엔 안전 모드:
migrate_table_schema(db_info, force_recreate=False)

✓ 테이블 스키마 준비 완료 (PK: hs_code, date_*, created_at)


## 4. 유틸리티 함수

In [4]:
def ensure_sorted_unique_dates(df: pd.DataFrame, date_col: str = "date") -> pd.DataFrame:
    """날짜 월말 정렬 + 중복 제거"""
    d = df.copy()
    d[date_col] = pd.to_datetime(d[date_col]) + MonthEnd(0)
    return (d.sort_values(date_col)
             .drop_duplicates([date_col])
             .reset_index(drop=True))


def _safe_value(v):
    """MySQL용 안전값 (NaN, inf, NaT → None)"""
    if v is None:
        return None
    if isinstance(v, pd.Timestamp):
        return v.to_pydatetime()
    if isinstance(v, float) and (math.isnan(v) or math.isinf(v)):
        return None
    try:
        if pd.isna(v):
            return None
    except (TypeError, ValueError):
        pass
    return v


def _to_json_params(metadata: Dict) -> str:
    """metadata를 JSON 문자열로 직렬화 (V5: 폭발 진단 필드 추가)"""
    safe = {
        "model":                 metadata.get("model"),
        "order":                 metadata.get("order"),
        "seasonal_order":        metadata.get("seasonal_order"),
        "aic":                   metadata.get("aic"),
        "bic":                   metadata.get("bic"),
        "aicc":                  metadata.get("aicc"),
        "use_log":               metadata.get("use_log"),
        "n_obs":                 metadata.get("n_obs"),
        "error":                 metadata.get("error"),
        # --- V5 진단 ---
        "n_clipped":             metadata.get("n_clipped"),
        "clip_ratio":            metadata.get("clip_ratio"),
        "max_fc_ratio":          metadata.get("max_fc_ratio"),
        "structural_break_flag": metadata.get("structural_break_flag"),
    }
    return json.dumps(safe, ensure_ascii=False, default=str)


print("✓ 유틸리티 함수 정의 완료 (V5)")

✓ 유틸리티 함수 정의 완료 (V5)


## 5. Robust SARIMA spec 탐색

V3의 144개 grid search 대신 monthly trade data에 적합한 **4개 후보**만 평가:

| 후보 | spec | 특징 |
|---|---|---|
| 1 | (0,1,1)(0,1,1,12) | Box-Jenkins Airline (default) |
| 2 | (1,1,1)(0,1,1,12) | + 비계절 AR(1) |
| 3 | (0,1,2)(0,1,1,12) | + 비계절 MA(2) |
| 4 | (1,1,1)(1,1,1,12) | full seasonal AR/MA |

모든 후보가 seasonal 성분(D=1, Q=1)을 강제 보유 → **평선 forecast 원천 차단**

In [5]:
def find_robust_sarima_params(
    y_train: pd.Series,
    raw_ref_max: float,
    use_log: bool = True,
    seasonal_period: int = 12,
    forecast_months: int = 18,
    ic: str = "aicc",
    hard_mult: float = 10.0,
) -> Tuple[tuple, tuple, dict, bool]:
    """
    Monthly trade data 전용 robust SARIMA spec 탐색.

    V5 변경:
    - sanity check를 raw(exp) 스케일에서 수행 → 폭발 spec을 '선택 단계'에서 배제
      (V4는 log 스케일에서 1e6 배수로 우회 → 폭발 spec이 그대로 선택됐음)
    - any_pass: 모든 후보가 폭발/실패했는지 여부 반환 (호출부에서 봉투 클립이 처리)

    Returns: best_order, best_sorder, fit_info, any_pass
    """
    candidates = [
        ((0, 1, 1), (0, 1, 1, seasonal_period)),  # Airline
        ((1, 1, 1), (0, 1, 1, seasonal_period)),  # + AR(1)
        ((0, 1, 2), (0, 1, 1, seasonal_period)),  # + MA(2)
        ((1, 1, 1), (1, 1, 1, seasonal_period)),  # full seasonal
    ]

    best_score  = np.inf
    best_order  = candidates[0][0]
    best_sorder = candidates[0][1]
    best_info   = {"aic": None, "bic": None, "aicc": None}
    any_pass    = False

    for order, sorder in candidates:
        try:
            fit = SARIMAX(
                y_train.astype(float),
                order=order,
                seasonal_order=sorder,
                enforce_stationarity=True,
                enforce_invertibility=True,
            ).fit(disp=False, maxiter=100)

            fcst     = fit.forecast(steps=forecast_months)
            fcst_raw = np.exp(fcst) if use_log else fcst

            # ★ raw 스케일 sanity: 폭발 spec 배제
            if not (fcst.notna().all()
                    and np.isfinite(fcst_raw).all()
                    and float(fcst_raw.max()) < raw_ref_max * hard_mult
                    and float(fcst_raw.min()) > 0):
                continue

            aic  = float(fit.aic)
            bic  = float(fit.bic)
            aicc = float(getattr(fit, "aicc", aic))
            score = {"aic": aic, "bic": bic}.get(ic.lower(), aicc)

            if np.isfinite(score) and score < best_score:
                best_score  = score
                best_order  = order
                best_sorder = sorder
                best_info   = {"aic": aic, "bic": bic, "aicc": aicc}
                any_pass    = True

        except Exception:
            continue

    return best_order, best_sorder, best_info, any_pass


print("✓ find_robust_sarima_params 정의 완료 (V5: raw 스케일 sanity)")

✓ find_robust_sarima_params 정의 완료 (V5: raw 스케일 sanity)


## 6. SARIMA 예측 메인 함수

로그 변환을 함수 내부로 흡수했으므로 **외부에서 raw 값 그대로** 전달하면 됩니다.

에러 케이스는 metadata에 분류 코드로 기록:
- `insufficient_obs`: 관측치 부족 (< min_obs)
- `non_positive_data`: 비양수 값 존재 (로그 변환 불가)
- `forecast_sanity_fail`: 예측값 NaN/inf/극단/음수
- `unexpected: <msg>`: 그 외

In [6]:
def run_sarima_forecast_monthly(
    df: pd.DataFrame,
    hs_code: str,
    forecast_months: int = 18,
    target_col: str = "exp_dlr",
    date_col: str = "date",
    ic: str = "aicc",
    min_obs: int = 36,
    use_log: bool = True,
    max_ann_growth: float = 1.5,   # ★ 봉투 상한 = 연 150% 성장 (이 이상은 폭발로 간주)
    hard_mult: float = 10.0,       # 최종 방어선 배수
) -> Tuple[pd.DataFrame, Dict]:
    """
    단일 HS 코드 월별 SARIMA 예측 (V5: 폭발 차단 봉투 클립)

    핵심 안전장치 (순서대로):
    1. find_robust_sarima_params: raw 스케일 sanity로 폭발 spec 배제
    2. ★ horizon-aware 봉투 클립: forecast를 hist_max×(1+g)^(h/12) 이내로 강제
       → 폭발(상단)·붕괴(하단) 동시 차단. 정상 종목은 봉투 안이라 그대로 통과.
    3. 최종 하드 가드: 클립 후에도 hist_max×hard_mult 초과 시 실측 복사(거의 미발동)

    Returns:
    --------
    result_df : ['date_month_end', 'expDlr', 'expDlr_forecast']
    metadata  : order/seasonal_order/aic/bic/aicc/use_log/n_obs/error
                + V5: n_clipped/clip_ratio/max_fc_ratio/structural_break_flag
    """
    metadata = {
        "hs_code": hs_code, "model": "SARIMA", "frequency": "monthly",
        "order": None, "seasonal_order": None,
        "aic": None, "bic": None, "aicc": None,
        "use_log": use_log, "n_obs": 0, "error": None,
        # V5 진단
        "n_clipped": 0, "clip_ratio": 0.0,
        "max_fc_ratio": None, "structural_break_flag": 0,
    }

    d = pd.DataFrame()
    try:
        d = ensure_sorted_unique_dates(df[[date_col, target_col]], date_col)
        d = d.rename(columns={date_col: "date_month_end"})

        y_raw = pd.Series(
            d[target_col].values,
            index=pd.to_datetime(d["date_month_end"])
        ).dropna()
        metadata["n_obs"] = int(len(y_raw))

        # 관측치 부족
        if len(y_raw) < min_obs:
            metadata["error"] = f"insufficient_obs: {len(y_raw)} < {min_obs}"
            d = d.rename(columns={target_col: "expDlr"})
            d["expDlr_forecast"] = d["expDlr"]
            return d, metadata

        # 양수 보장 (로그용)
        if use_log and (y_raw <= 0).any():
            metadata["error"] = f"non_positive_data: {int((y_raw <= 0).sum())}"
            d = d.rename(columns={target_col: "expDlr"})
            d["expDlr_forecast"] = d["expDlr"]
            return d, metadata

        # 로그 변환 (내부 스케일)
        y = np.log(y_raw) if use_log else y_raw
        hist_max = float(y_raw.max())
        hist_min = float(y_raw.min())

        # robust spec 탐색 (raw 스케일 sanity)
        order, sorder, fit_info, any_pass = find_robust_sarima_params(
            y, raw_ref_max=hist_max, use_log=use_log,
            seasonal_period=12, forecast_months=forecast_months,
            ic=ic, hard_mult=hard_mult,
        )

        # 최종 fit
        fit = SARIMAX(
            y, order=order, seasonal_order=sorder,
            enforce_stationarity=True, enforce_invertibility=True,
        ).fit(disp=False, maxiter=100)
        fc_inner = fit.forecast(steps=forecast_months).reset_index(drop=True)

        # ★★★ 핵심: horizon-aware 봉투 클립 (폭발/붕괴 차단) ★★★
        h         = np.arange(1, forecast_months + 1)
        max_slope = np.log1p(max_ann_growth) / 12.0   # 월 단위 최대 로그 기울기
        if use_log:
            ceil_in  = np.log(hist_max) + max_slope * h
            floor_in = np.log(max(hist_min, 1.0)) - max_slope * h
        else:
            gf       = np.power(1.0 + max_ann_growth, h / 12.0)
            ceil_in  = hist_max * gf
            floor_in = np.maximum(hist_min / gf, 0.0)

        vals     = fc_inner.values.astype(float)
        clipped  = np.clip(vals, floor_in, ceil_in)
        n_clip   = int((vals > ceil_in).sum() + (vals < floor_in).sum())
        forecast = pd.Series(np.exp(clipped) if use_log else clipped)

        metadata["n_clipped"]    = n_clip
        metadata["clip_ratio"]   = round(n_clip / forecast_months, 3)
        metadata["max_fc_ratio"] = round(float(forecast.max()) / hist_max, 3)
        # 절반 이상 clip → 모델이 폭발 시도 = 구조변화/M&A 의심 → 사후 검토 대상
        metadata["structural_break_flag"] = int(n_clip >= forecast_months * 0.5)

        # 최종 하드 가드 (클립 후엔 사실상 미발동, 방어선)
        if not (forecast.notna().all()
                and np.isfinite(forecast).all()
                and float(forecast.max()) < hist_max * hard_mult
                and float(forecast.min()) > 0):
            metadata["error"] = (
                f"forecast_sanity_fail_after_clip "
                f"(max={float(forecast.max()):.2e}, hist_max={hist_max:.2e})"
            )
            d = d.rename(columns={target_col: "expDlr"})
            d["expDlr_forecast"] = d["expDlr"]
            return d, metadata

        # 미래 날짜
        last_date    = y_raw.index.max()
        future_dates = pd.date_range(last_date + MonthEnd(1),
                                     periods=forecast_months, freq="M")

        # 결과 조립
        d = d.rename(columns={target_col: "expDlr"})
        d["expDlr_forecast"] = d["expDlr"]
        future_rows = pd.DataFrame({
            "date_month_end":  future_dates,
            "expDlr":          np.nan,
            "expDlr_forecast": forecast.values.astype(float),
        })
        d = pd.concat([d, future_rows], ignore_index=True)
        d = d.sort_values("date_month_end").reset_index(drop=True)

        metadata.update({
            "order":          list(order),
            "seasonal_order": list(sorder),
            "aic":            fit_info["aic"],
            "bic":            fit_info["bic"],
            "aicc":           fit_info["aicc"],
        })
        return d, metadata

    except Exception as e:
        metadata["error"] = f"unexpected: {str(e)[:200]}"
        if not d.empty and target_col in d.columns:
            d = d.rename(columns={target_col: "expDlr"})
            if "expDlr_forecast" not in d.columns:
                d["expDlr_forecast"] = d["expDlr"]
            return d, metadata
        return df.copy(), metadata


print("✓ run_sarima_forecast_monthly 정의 완료 (V5: 봉투 클립)")

✓ run_sarima_forecast_monthly 정의 완료 (V5: 봉투 클립)


## 7. 분기별 집계

In [7]:
def aggregate_to_quarter(monthly_df: pd.DataFrame) -> pd.DataFrame:
    """
    월별 → 분기별 합계 집계
    
    Input  : date_month_end, expDlr, expDlr_forecast
    Output : date_quarter_end, expDlr, expDlr_forecast, is_forecast
    """
    df = monthly_df.copy()
    df["date_month_end"] = pd.to_datetime(df["date_month_end"])
    df["quarter"] = df["date_month_end"].dt.to_period("Q")
    
    quarterly = df.groupby("quarter", as_index=False).agg(
        expDlr          = ("expDlr",          lambda x: x.sum(min_count=1)),
        expDlr_forecast = ("expDlr_forecast", lambda x: x.sum(min_count=1)),
    )
    
    quarterly["is_forecast"]      = quarterly["expDlr"].isna().astype(int)
    quarterly["date_quarter_end"] = (quarterly["quarter"].dt.to_timestamp(how="end")
                                     + MonthEnd(0))
    
    return quarterly[["date_quarter_end", "expDlr", "expDlr_forecast", "is_forecast"]]


print("✓ aggregate_to_quarter 정의 완료")

✓ aggregate_to_quarter 정의 완료


## 8. DB 저장 (단일 HS 단위 즉시 UPSERT)

`(hs_code, date_month_end, created_at)` PK 기반 `ON DUPLICATE KEY UPDATE` upsert.

장점:
- 단일 HS 처리 후 바로 commit → peak 메모리 최소화
- 같은 run을 재실행해도 안전 (upsert)
- 일별 변화 추적은 `created_at` 다른 row가 누적되어 자연 지원

In [8]:
def save_single_hs_to_db(
    hs_code:    str,
    monthly_df: pd.DataFrame,
    quarter_df: pd.DataFrame,
    metadata:   Dict,
    created_at: datetime,
    db_info:    Dict,
) -> Tuple[int, int]:
    """단일 HS 코드 월별 + 분기별 결과를 즉시 DB upsert"""
    params_json = _to_json_params(metadata)
    
    # 월별 row
    monthly_rows = []
    for _, row in monthly_df.iterrows():
        monthly_rows.append((
            hs_code,
            _safe_value(row["date_month_end"]),
            _safe_value(row.get("expDlr")),
            _safe_value(row.get("expDlr_forecast")),
            int(pd.isna(row.get("expDlr"))),
            params_json,
            created_at,
        ))
    
    # 분기별 row
    quarter_rows = []
    for _, row in quarter_df.iterrows():
        quarter_rows.append((
            hs_code,
            _safe_value(row["date_quarter_end"]),
            _safe_value(row.get("expDlr")),
            _safe_value(row.get("expDlr_forecast")),
            int(row.get("is_forecast", 0)),
            params_json,
            created_at,
        ))
    
    sql_m = """
    INSERT INTO us_trade_export_monthly_with_forecast
        (hs_code, date_month_end, expDlr, expDlr_forecast,
         is_forecast, params, created_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        expDlr          = VALUES(expDlr),
        expDlr_forecast = VALUES(expDlr_forecast),
        is_forecast     = VALUES(is_forecast),
        params          = VALUES(params)
    """
    
    sql_q = """
    INSERT INTO us_trade_export_quarter_with_forecast
        (hs_code, date_quarter_end, expDlr, expDlr_forecast,
         is_forecast, params, created_at)
    VALUES (%s, %s, %s, %s, %s, %s, %s)
    ON DUPLICATE KEY UPDATE
        expDlr          = VALUES(expDlr),
        expDlr_forecast = VALUES(expDlr_forecast),
        is_forecast     = VALUES(is_forecast),
        params          = VALUES(params)
    """
    
    conn = pymysql.connect(**db_info)
    try:
        with conn.cursor() as cursor:
            if monthly_rows:
                cursor.executemany(sql_m, monthly_rows)
            if quarter_rows:
                cursor.executemany(sql_q, quarter_rows)
        conn.commit()
    finally:
        conn.close()
    
    return len(monthly_rows), len(quarter_rows)


def save_error_only_to_db(
    hs_code:    str,
    metadata:   Dict,
    created_at: datetime,
    db_info:    Dict,
):
    """
    예측 실패 케이스도 DB에 1개 row 저장 (에러 추적 목적).
    더미 row: date_month_end = 1900-01-31, expDlr = NULL
    """
    sql = """
    INSERT INTO us_trade_export_monthly_with_forecast
        (hs_code, date_month_end, expDlr, expDlr_forecast,
         is_forecast, params, created_at)
    VALUES (%s, '1900-01-31', NULL, NULL, 1, %s, %s)
    ON DUPLICATE KEY UPDATE
        params = VALUES(params)
    """
    conn = pymysql.connect(**db_info)
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql, (hs_code, _to_json_params(metadata), created_at))
        conn.commit()
    finally:
        conn.close()


print("✓ DB 저장 함수 정의 완료")

✓ DB 저장 함수 정의 완료


## 9. 배치 실행 (체크포인트 + 즉시 저장)

`done_hs_codes.txt`로 체크포인트 관리:
- `skip_done=False` (기본): 모든 HS code 재예측 (덮어쓰기 = upsert)
- `skip_done=True`: 이미 처리된 코드 스킵 (중단 후 재시작 시 사용)

In [9]:
def forecast_all_hs_codes(
    trade_data:      pd.DataFrame,
    db_info:         Dict,
    forecast_months: int = 18,
    min_obs:         int = 36,
    ic:              str = "aicc",
    use_log:         bool = True,
    max_ann_growth:  float = 1.5,   # ★ V5 봉투 상한 (연 성장률)
    hard_mult:       float = 10.0,
    date_col:        str = "date",
    target_col:      str = "exp_dlr",
    checkpoint_path: str = "done_hs_codes.txt",
    skip_done:       bool = False,
    save_errors:     bool = True,
) -> Dict:
    """
    모든 HS 코드 SARIMA 예측 + 단일 단위 DB upsert. (V5: 봉투 클립 + 구조변화 집계)
    """
    created_at = datetime.now()

    done_set = set()
    if skip_done and os.path.exists(checkpoint_path):
        with open(checkpoint_path, "r", encoding="utf-8") as f:
            done_set = {line.strip() for line in f if line.strip()}
        print(f"  체크포인트 로드: {len(done_set):,}개 코드 이미 처리됨")

    hs_codes = trade_data["hs_code"].unique()
    total    = len(hs_codes)

    print(f"총 {total:,}개 HS 코드")
    print(f"  forecast_months={forecast_months}, min_obs={min_obs}, ic={ic}, use_log={use_log}")
    print(f"  max_ann_growth={max_ann_growth} (봉투 상한), hard_mult={hard_mult}")
    print(f"  created_at={created_at}")
    print(f"  checkpoint={checkpoint_path}, skip_done={skip_done}")
    print("=" * 80)

    stats = {
        "ok": 0, "skipped": 0,
        "error_insufficient": 0, "error_non_positive": 0,
        "error_sanity":       0, "error_unexpected":   0,
        "clipped_any":        0, "structural_break":   0,
        "errors": [], "spec_distribution": {},
    }

    pbar = tqdm(hs_codes, desc="예측", unit="HS")
    for hs_code in pbar:
        if hs_code in done_set:
            stats["skipped"] += 1
            continue

        pbar.set_postfix({
            "현재": str(hs_code)[:10],
            "OK":   stats["ok"],
            "BRK":  stats["structural_break"],
            "ERR":  (stats["error_insufficient"] + stats["error_non_positive"]
                     + stats["error_sanity"]    + stats["error_unexpected"]),
        })

        try:
            hs_data = trade_data[trade_data["hs_code"] == hs_code].copy()

            result_df, metadata = run_sarima_forecast_monthly(
                df=hs_data,
                hs_code=hs_code,
                forecast_months=forecast_months,
                target_col=target_col,
                date_col=date_col,
                ic=ic,
                min_obs=min_obs,
                use_log=use_log,
                max_ann_growth=max_ann_growth,
                hard_mult=hard_mult,
            )

            err = metadata.get("error")
            if err:
                if   err.startswith("insufficient"):     stats["error_insufficient"] += 1
                elif err.startswith("non_positive"):     stats["error_non_positive"] += 1
                elif err.startswith("forecast_sanity"):  stats["error_sanity"]       += 1
                else:                                    stats["error_unexpected"]   += 1

                stats["errors"].append({"hs_code": hs_code, "error": err})

                if save_errors:
                    save_error_only_to_db(hs_code, metadata, created_at, db_info)

                with open(checkpoint_path, "a", encoding="utf-8") as f:
                    f.write(f"{hs_code}\n")
                continue

            # 분기 집계
            quarter_df = aggregate_to_quarter(result_df)

            # DB upsert
            save_single_hs_to_db(
                hs_code=hs_code,
                monthly_df=result_df,
                quarter_df=quarter_df,
                metadata=metadata,
                created_at=created_at,
                db_info=db_info,
            )

            spec_key = f"{tuple(metadata['order'])}{tuple(metadata['seasonal_order'])}"
            stats["spec_distribution"][spec_key] = stats["spec_distribution"].get(spec_key, 0) + 1

            if metadata.get("n_clipped", 0) > 0:           stats["clipped_any"]      += 1
            if metadata.get("structural_break_flag", 0):   stats["structural_break"] += 1

            stats["ok"] += 1

            with open(checkpoint_path, "a", encoding="utf-8") as f:
                f.write(f"{hs_code}\n")

        except Exception as e:
            stats["error_unexpected"] += 1
            stats["errors"].append({"hs_code": hs_code, "error": f"unexpected: {str(e)[:200]}"})
            continue

    pbar.close()

    print("=" * 80)
    print(f"\n[처리 결과] created_at={created_at}")
    print(f"  성공            : {stats['ok']:,}")
    print(f"  스킵 (done)     : {stats['skipped']:,}")
    print(f"  실패 (관측↓)    : {stats['error_insufficient']:,}")
    print(f"  실패 (음수)     : {stats['error_non_positive']:,}")
    print(f"  실패 (sanity)   : {stats['error_sanity']:,}")
    print(f"  실패 (기타)     : {stats['error_unexpected']:,}")
    print(f"  ─ 봉투 클립 발생 : {stats['clipped_any']:,}  (일부 월 forecast 제한됨)")
    print(f"  ─ 구조변화 플래그: {stats['structural_break']:,}  (★ 사후 검토 권장: M&A/급등 의심)")

    if stats["spec_distribution"]:
        print(f"\n[채택 spec 분포 상위 5개]")
        for spec, cnt in sorted(stats["spec_distribution"].items(), key=lambda x: -x[1])[:5]:
            print(f"  {spec}: {cnt:,}")

    return stats


print("✓ forecast_all_hs_codes 정의 완료 (V5)")

✓ forecast_all_hs_codes 정의 완료 (V5)


## 10. 원본 데이터 로드

In [10]:
def load_trade_data(db_info: Dict) -> pd.DataFrame:
    """us_export_data 테이블에서 raw 수출 데이터 로드"""
    sql = """
    SELECT hs_code, date, exp_dlr
    FROM us_export_data
    WHERE exp_dlr IS NOT NULL
    ORDER BY hs_code, date
    """
    
    conn = pymysql.connect(**db_info)
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql)
            rows = cursor.fetchall()
            cols = [d[0] for d in cursor.description]
    finally:
        conn.close()
    
    df = pd.DataFrame(rows, columns=cols)
    df["date"] = pd.to_datetime(df["date"])
    return df


trade_df = load_trade_data(db_info)
print(f"데이터 로드: {len(trade_df):,} rows")
print(f"HS 코드 수: {trade_df['hs_code'].nunique():,}")
print(f"기간       : {trade_df['date'].min()} ~ {trade_df['date'].max()}")
print()
print(trade_df.head())

데이터 로드: 60,094 rows
HS 코드 수: 500
기간       : 2016-01-31 00:00:00 ~ 2026-05-31 00:00:00

  hs_code       date    exp_dlr
0  020130 2016-01-31  154815020
1  020130 2016-02-29  161201010
2  020130 2016-03-31  190582390
3  020130 2016-04-30  199209993
4  020130 2016-05-31  216155091


## 11. 단일 HS 코드 테스트 (전체 실행 전 필수)

이전에 평선이 나왔던 사례 `851779`로 검증:

| 항목 | 기대값 |
|---|---|
| 채택 spec | seasonal 성분(D=1, Q=1) 포함 |
| Error | None |
| 변동계수(CV) | > 0.05 |
| Min/Max ratio | < 0.95 |
| All positive | True |

In [11]:
TEST_HS = "851779"   # CSV에서 rank 1로 폭발했던 종목

hs_data = trade_df[trade_df["hs_code"] == TEST_HS].copy()
print(f"테스트 HS: {TEST_HS} / 관측치: {len(hs_data)}")
print(f"기간: {hs_data['date'].min()} ~ {hs_data['date'].max()}")
print()

result_df, meta = run_sarima_forecast_monthly(
    df=hs_data,
    hs_code=TEST_HS,
    forecast_months=18,
    target_col="exp_dlr",
    date_col="date",
    ic="aicc",
    min_obs=36,
    use_log=True,
    max_ann_growth=1.5,
)

print(f"채택 spec  : {tuple(meta['order'])}{tuple(meta['seasonal_order'])}"
      if meta["order"] else "  → 모델 fitting 실패")
print(f"AIC / BIC / AICc : "
      f"{meta['aic']:.2f} / {meta['bic']:.2f} / {meta['aicc']:.2f}"
      if meta["aic"] is not None else "  → 모델 fitting 실패")
print(f"Error      : {meta['error']}")
print(f"n_obs      : {meta['n_obs']}")
print()
print(f"[V5 폭발 진단]")
print(f"  max_fc_ratio          : {meta['max_fc_ratio']}  (forecast_max / hist_max)")
print(f"  n_clipped / 18        : {meta['n_clipped']}")
print(f"  clip_ratio            : {meta['clip_ratio']}")
print(f"  structural_break_flag : {meta['structural_break_flag']}  "
      f"{'★ 구조변화 의심 → 사후 검토' if meta['structural_break_flag'] else ''}")

# 평선/폭발 검증
fcst_only = result_df[result_df["expDlr"].isna()]["expDlr_forecast"].head(12)
if len(fcst_only) > 0 and fcst_only.mean() > 0:
    cv  = fcst_only.std() / fcst_only.mean()
    mmr = fcst_only.min() / fcst_only.max()
    print(f"\n[Forecast 검증 - 12개월]")
    print(f"  Mean         : ${fcst_only.mean():,.0f}")
    print(f"  Max          : ${fcst_only.max():,.0f}   (폭발 아님 확인: hist_max 대비 {meta['max_fc_ratio']}배)")
    print(f"  CV (변동계수): {cv:.4f}  {'✓' if cv > 0.05 else '✗ 평선'}")
    print(f"  Min/Max ratio: {mmr:.3f}  {'✓' if mmr < 0.95 else '✗ 평선'}")
    print(f"  All positive : {bool((fcst_only > 0).all())}")

테스트 HS: 851779 / 관측치: 53
기간: 2022-01-31 00:00:00 ~ 2026-05-31 00:00:00

채택 spec  : (0, 1, 1)(0, 1, 1, 12)
AIC / BIC / AICc : -7.33 / -2.26 / -6.66
Error      : None
n_obs      : 53

[V5 폭발 진단]
  max_fc_ratio          : 0.612  (forecast_max / hist_max)
  n_clipped / 18        : 0
  clip_ratio            : 0.0
  structural_break_flag : 0  

[Forecast 검증 - 12개월]
  Mean         : $124,370,972
  Max          : $133,583,447   (폭발 아님 확인: hist_max 대비 0.612배)
  CV (변동계수): 0.0434  ✗ 평선
  Min/Max ratio: 0.861  ✓
  All positive : True


## 12. 전체 HS 코드 예측 + DB 저장

`skip_done=False`로 모든 HS code 재예측 (PK 덕분에 upsert로 안전).
중단 후 재시작 시 `skip_done=True`로 변경.

In [12]:
# 첫 실행 시: 체크포인트 파일 초기화 (선택)
CHECKPOINT = "done_hs_codes.txt"
if os.path.exists(CHECKPOINT):
    os.remove(CHECKPOINT)
    print(f"  기존 체크포인트 삭제: {CHECKPOINT}")

stats = forecast_all_hs_codes(
    trade_data      = trade_df,
    db_info         = db_info,
    forecast_months = 18,
    min_obs         = 36,
    ic              = "aicc",
    use_log         = True,
    date_col        = "date",
    target_col      = "exp_dlr",
    checkpoint_path = CHECKPOINT,
    skip_done       = False,
    save_errors     = True,
)

  기존 체크포인트 삭제: done_hs_codes.txt
총 500개 HS 코드
  forecast_months=18, min_obs=36, ic=aicc, use_log=True
  max_ann_growth=1.5 (봉투 상한), hard_mult=10.0
  created_at=2026-07-23 23:44:35.395245
  checkpoint=done_hs_codes.txt, skip_done=False


예측:   0%|          | 0/500 [00:00<?, ?HS/s]


[처리 결과] created_at=2026-07-23 23:44:35.395245
  성공            : 498
  스킵 (done)     : 0
  실패 (관측↓)    : 0
  실패 (음수)     : 2
  실패 (sanity)   : 0
  실패 (기타)     : 0
  ─ 봉투 클립 발생 : 3  (일부 월 forecast 제한됨)
  ─ 구조변화 플래그: 0  (★ 사후 검토 권장: M&A/급등 의심)

[채택 spec 분포 상위 5개]
  (0, 1, 1)(0, 1, 1, 12): 258
  (0, 1, 2)(0, 1, 1, 12): 110
  (1, 1, 1)(0, 1, 1, 12): 100
  (1, 1, 1)(1, 1, 1, 12): 30


## 13. 결과 진단

3가지 진단 SQL로 검증:
1. **에러 분포** — 어떤 에러가 몇 개 HS code에서 발생했는지
2. **채택 spec 분포** — 모델 후보 4개가 골고루 선택되는지 (한쪽에 쏠리면 후보 부족 신호)
3. **평선 의심 진단** — CV가 비정상적으로 낮은 케이스 자동 탐지

In [13]:
import datetime as dt

cutoff = dt.datetime.now() - dt.timedelta(hours=6)  # 최근 6시간 내 run

sql_errors = """
SELECT
    JSON_UNQUOTE(JSON_EXTRACT(params, '$.error')) AS error_type,
    COUNT(DISTINCT hs_code) AS n_hs
FROM us_trade_export_monthly_with_forecast
WHERE created_at >= %s
  AND JSON_EXTRACT(params, '$.error') IS NOT NULL
GROUP BY error_type
ORDER BY n_hs DESC
"""

sql_specs = """
SELECT
    JSON_UNQUOTE(JSON_EXTRACT(params, '$.order'))          AS order_spec,
    JSON_UNQUOTE(JSON_EXTRACT(params, '$.seasonal_order')) AS sorder_spec,
    COUNT(DISTINCT hs_code) AS n_hs
FROM us_trade_export_monthly_with_forecast
WHERE created_at >= %s
  AND (JSON_EXTRACT(params, '$.error') IS NULL
       OR JSON_UNQUOTE(JSON_EXTRACT(params, '$.error')) = 'null')
GROUP BY order_spec, sorder_spec
ORDER BY n_hs DESC
"""

sql_flat = """
SELECT
    hs_code,
    AVG(expDlr_forecast)                        AS mean_fcst,
    STD(expDlr_forecast) / AVG(expDlr_forecast) AS cv,
    COUNT(*) AS n_fcst
FROM us_trade_export_monthly_with_forecast
WHERE is_forecast = 1
  AND created_at >= %s
  AND expDlr_forecast IS NOT NULL
  AND date_month_end > '2000-01-01'
GROUP BY hs_code
HAVING cv < 0.01 AND n_fcst >= 12
ORDER BY cv ASC
LIMIT 20
"""

conn = pymysql.connect(**db_info)
try:
    with conn.cursor() as cursor:
        print("=" * 70)
        print("[1] 에러 분포")
        print("=" * 70)
        cursor.execute(sql_errors, (cutoff,))
        rows = cursor.fetchall()
        if rows:
            df_err = pd.DataFrame(rows, columns=["error_type", "n_hs"])
            print(df_err.to_string(index=False))
        else:
            print("✓ 에러 없음")
        
        print("\n" + "=" * 70)
        print("[2] 채택 spec 분포")
        print("=" * 70)
        cursor.execute(sql_specs, (cutoff,))
        rows = cursor.fetchall()
        if rows:
            df_spec = pd.DataFrame(rows, columns=["order", "seasonal_order", "n_hs"])
            print(df_spec.to_string(index=False))
        else:
            print("(데이터 없음)")
        
        print("\n" + "=" * 70)
        print("[3] 평선 의심 HS 코드 (CV < 0.01)")
        print("=" * 70)
        cursor.execute(sql_flat, (cutoff,))
        rows = cursor.fetchall()
        if rows:
            df_flat = pd.DataFrame(rows, columns=["hs_code", "mean_fcst", "cv", "n_fcst"])
            print(df_flat.to_string(index=False))
            print(f"\n⚠️ {len(rows)}개 HS code에서 평선 의심됨 — 추가 진단 필요")
        else:
            print("✓ 평선 forecast 없음 — 모델 정상 작동")
finally:
    conn.close()

[1] 에러 분포
✓ 에러 없음

[2] 채택 spec 분포
order seasonal_order  n_hs
 None           None   500

[3] 평선 의심 HS 코드 (CV < 0.01)
✓ 평선 forecast 없음 — 모델 정상 작동


## 14. 특정 HS 결과 확인

In [14]:
def check_forecast_db(hs_code: str, db_info: Dict):
    """특정 HS 코드의 최신 run 결과를 월별/분기별로 가져오기"""
    sql_m = """
    SELECT date_month_end, expDlr, expDlr_forecast, is_forecast, params, created_at
    FROM us_trade_export_monthly_with_forecast
    WHERE hs_code = %s
      AND date_month_end > '2000-01-01'
      AND created_at = (
          SELECT MAX(created_at) FROM us_trade_export_monthly_with_forecast
          WHERE hs_code = %s
      )
    ORDER BY date_month_end
    """
    
    sql_q = """
    SELECT date_quarter_end, expDlr, expDlr_forecast, is_forecast, created_at
    FROM us_trade_export_quarter_with_forecast
    WHERE hs_code = %s
      AND created_at = (
          SELECT MAX(created_at) FROM us_trade_export_quarter_with_forecast
          WHERE hs_code = %s
      )
    ORDER BY date_quarter_end
    """
    
    conn = pymysql.connect(**db_info)
    try:
        with conn.cursor() as cursor:
            cursor.execute(sql_m, (hs_code, hs_code))
            rows_m = cursor.fetchall()
            cols_m = [d[0] for d in cursor.description]
            df_m = pd.DataFrame(rows_m, columns=cols_m)
            
            cursor.execute(sql_q, (hs_code, hs_code))
            rows_q = cursor.fetchall()
            cols_q = [d[0] for d in cursor.description]
            df_q = pd.DataFrame(rows_q, columns=cols_q)
    finally:
        conn.close()
    
    return df_m, df_q


df_monthly, df_quarter = check_forecast_db("854232", db_info)
print(f"월별 row: {len(df_monthly)} / 분기별 row: {len(df_quarter)}")
print()
print("[월별 마지막 18개]")
print(df_monthly.tail(18).to_string(index=False))

KeyboardInterrupt: 

In [33]:
# 다른 HS 코드도 확인
for hs in ["851779", "340242"]:
    df_m, _ = check_forecast_db(hs, db_info)
    if df_m.empty:
        print(f"{hs}: 데이터 없음")
        continue
    
    fcst = df_m[df_m["is_forecast"] == 1]["expDlr_forecast"].head(12)
    if len(fcst) > 0 and fcst.mean() > 0:
        cv = fcst.std() / fcst.mean()
        params = df_m["params"].iloc[-1]
        print(f"\n[{hs}]")
        print(f"  forecast 12개월 평균: ${fcst.mean():,.0f}")
        print(f"  CV: {cv:.4f}")
        print(f"  params: {params[:80]}")
    else:
        params = df_m["params"].iloc[-1]
        print(f"\n[{hs}] forecast 없음 (에러일 가능성)")
        print(f"  params: {params}")


[851779]
  forecast 12개월 평균: $121,158,393
  CV: 0.0428
  params: {"model": "SARIMA", "order": [0, 1, 1], "seasonal_order": [0, 1, 1, 12], "aic": 

[340242]
  forecast 12개월 평균: $65,427,485
  CV: 0.0807
  params: {"model": "SARIMA", "order": [0, 1, 1], "seasonal_order": [0, 1, 1, 12], "aic": 


## 15. 스테일(폭발) 데이터 정리 + ranking 쿼리 가이드

**중요**: 저장은 `(hs_code, date_month_end, created_at)` 3중 PK로 UPSERT 되므로,
V5 재예측은 **새 `created_at`** 으로 들어가고 **과거 V3/V4 run의 폭발 행은 그대로 남습니다.**
따라서 다음 중 하나가 반드시 필요합니다:

1. **(권장)** ranking/조회 쿼리에서 항상 **최신 `created_at` 만** 읽기 — 아래 템플릿 참고
2. 또는 과거 폭발 행을 물리적으로 삭제 (아래 cleanup 셀)

CSV가 엉망이었던 직접 원인: ranking 쿼리가 **과거 폭발 run의 forecast(예: `expDlr_forecast=e+158`)를
trailing-12M 합산에 섞어 넣었기** 때문. `recent_12month`/`future_12month` 자체가 폭발값이 되어
`growth_rate`가 `e+42`까지 치솟음.

In [34]:
# ── (A) DB에 남은 폭발 행 진단 ──────────────────────────────
# expDlr_forecast의 절대 규모로 비정상값 탐지
sql_explode = """
SELECT hs_code,
       COUNT(*)                              AS n_rows,
       MAX(expDlr_forecast)                  AS max_fc,
       MAX(created_at)                       AS latest_run
FROM us_trade_export_monthly_with_forecast
WHERE expDlr_forecast > 1e13          -- 10조 달러 초과 = 명백한 폭발
GROUP BY hs_code
ORDER BY max_fc DESC
LIMIT 30;
"""
conn = pymysql.connect(**db_info)
try:
    with conn.cursor() as cur:
        cur.execute(sql_explode)
        rows = cur.fetchall()
        cols = [c[0] for c in cur.description]
finally:
    conn.close()

explode_df = pd.DataFrame(rows, columns=cols)
print(f"폭발 의심 HS code(>1e13): {len(explode_df)}개")
print(explode_df.to_string(index=False) if len(explode_df) else "  → 없음 (clean)")

폭발 의심 HS code(>1e13): 18개
hs_code  n_rows        max_fc          latest_run
 340242       7 7.546638e+219 2025-12-13 23:00:31
 854149      25 2.053816e+212 2026-01-22 17:48:20
 300251      16 4.197696e+206 2026-01-22 17:48:20
 851779      18 1.658955e+206 2026-01-22 17:48:20
 842132      16 4.500995e+195 2026-01-22 17:48:20
 290351       7 1.514511e+191 2025-12-13 23:00:31
 852589       9 4.508563e+179 2026-01-22 17:48:20
 902789       7 5.094528e+176 2025-12-13 23:00:31
 854911       9 7.261487e+152 2026-01-22 17:48:20
 902781      25 6.190837e+145 2026-01-22 17:48:20
 300242       9 5.942472e+131 2026-01-22 17:48:20
 854141      23 1.818817e+130 2026-01-22 17:48:20
 293149      15 3.231350e+122 2026-01-22 17:48:20
 340250      32 2.213318e+122 2026-01-22 17:48:20
 390729      18 2.671419e+113 2026-01-22 17:48:20
 870451      18 9.613047e+100 2026-01-22 17:48:20
 851713      14  9.053310e+80 2025-12-13 23:00:31
 870121      14  3.026985e+67 2025-12-13 23:00:31


In [35]:
# ── (B) 과거 폭발 run 정리 (선택) ───────────────────────────
# 방법 1: 특정 created_at(과거 run)보다 오래된 행 전체 삭제
#         → 최신 V5 run만 남김. CUTOFF를 최신 run 직전 시각으로 지정.
#
# CUTOFF = "2026-06-01 00:00:00"   # 이 시각 이전 run 삭제
# for tbl in ["us_trade_export_monthly_with_forecast",
#             "us_trade_export_quarter_with_forecast"]:
#     conn = pymysql.connect(**db_info)
#     try:
#         with conn.cursor() as cur:
#             cur.execute(f"DELETE FROM {tbl} WHERE created_at < %s", (CUTOFF,))
#             print(f"{tbl}: {cur.rowcount:,} rows 삭제")
#         conn.commit()
#     finally:
#         conn.close()

print("※ 삭제는 주석 해제 후 신중히 실행. 기본은 비활성 상태.")
print("  권장: 삭제 대신 아래 (C) ranking 쿼리로 '최신 created_at만' 읽기.")

※ 삭제는 주석 해제 후 신중히 실행. 기본은 비활성 상태.
  권장: 삭제 대신 아래 (C) ranking 쿼리로 '최신 created_at만' 읽기.


In [36]:
# ── (C) 폭발-안전 ranking 쿼리 템플릿 ───────────────────────
# 핵심: ① HS별 최신 created_at만 사용  ② structural_break는 별도 플래그로 분리
#       ③ trailing-12M 합산은 그 최신 run의 expDlr_forecast로만 계산
PAST_DATE   = "2026-05-31"   # recent_12M 종료월
FUTURE_DATE = "2026-06-30"   # future_12M 종료월

sql_rank = """
WITH latest AS (                       -- HS별 최신 run
    SELECT hs_code, MAX(created_at) AS ca
    FROM us_trade_export_monthly_with_forecast
    GROUP BY hs_code
),
m AS (                                 -- 최신 run의 월별 데이터만
    SELECT t.hs_code, t.date_month_end, t.expDlr_forecast,
           JSON_EXTRACT(t.params, '$.structural_break_flag') AS brk,
           JSON_EXTRACT(t.params, '$.max_fc_ratio')          AS max_fc_ratio
    FROM us_trade_export_monthly_with_forecast t
    JOIN latest l
      ON t.hs_code = l.hs_code AND t.created_at = l.ca
)
SELECT
    hs_code,
    SUM(CASE WHEN date_month_end >  DATE_SUB(%s, INTERVAL 12 MONTH)
              AND date_month_end <= %s THEN expDlr_forecast END) AS recent_12month,
    SUM(CASE WHEN date_month_end >  DATE_SUB(%s, INTERVAL 12 MONTH)
              AND date_month_end <= %s THEN expDlr_forecast END) AS future_12month,
    MAX(brk)          AS structural_break_flag,
    MAX(max_fc_ratio) AS max_fc_ratio
FROM m
GROUP BY hs_code
HAVING recent_12month > 0 AND future_12month > 0
"""
conn = pymysql.connect(**db_info)
try:
    with conn.cursor() as cur:
        cur.execute(sql_rank, (PAST_DATE, PAST_DATE, FUTURE_DATE, FUTURE_DATE))
        rows = cur.fetchall()
        cols = [c[0] for c in cur.description]
finally:
    conn.close()

rank_df = pd.DataFrame(rows, columns=cols)
rank_df["growth_rate"] = (rank_df["future_12month"] / rank_df["recent_12month"] - 1.0) * 100.0

# 폭발 방어: 비정상 규모/구조변화 분리
rank_df["structural_break_flag"] = rank_df["structural_break_flag"].fillna(0).astype(int)
clean = rank_df[rank_df["structural_break_flag"] == 0].copy()       # 안정 종목
review = rank_df[rank_df["structural_break_flag"] == 1].copy()      # 검토 대상(M&A/급등)

clean = clean.sort_values("growth_rate", ascending=False).reset_index(drop=True)
clean.insert(0, "rank", clean.index + 1)

print(f"전체 {len(rank_df)} / 안정 {len(clean)} / 검토대상 {len(review)}")
print("\n[안정 종목 상위 15 — 폭발 없음 확인]")
print(clean.head(15)[["rank","hs_code","recent_12month","future_12month","growth_rate"]]
      .to_string(index=False))
print(f"\n[검토 대상(structural_break) {len(review)}개 — 별도 컨센서스 override 권장]")
print(review.sort_values("max_fc_ratio", ascending=False)
      .head(10)[["hs_code","growth_rate","max_fc_ratio"]].to_string(index=False))

전체 498 / 안정 498 / 검토대상 0

[안정 종목 상위 15 — 폭발 없음 확인]
 rank hs_code  recent_12month  future_12month  growth_rate
    1  854159    2.945961e+09    3.435088e+09    16.603292
    2  711291    4.080849e+09    4.629034e+09    13.433109
    3  851680    6.218497e+08    6.880629e+08    10.647795
    4  271020    9.507833e+08    1.040282e+09     9.413124
    5  852349    7.869915e+08    8.603934e+08     9.326902
    6  270900    9.895720e+10    1.079925e+11     9.130514
    7  711011    2.709373e+09    2.944296e+09     8.670783
    8  854449    3.193155e+09    3.459916e+09     8.354147
    9  870451    9.538340e+08    1.032384e+09     8.235198
   10  847330    4.275455e+10    4.625881e+10     8.196217
   11  870421    1.224881e+09    1.323280e+09     8.033377
   12  847150    3.883341e+10    4.188653e+10     7.862072
   13  930690    5.776643e+09    6.189522e+09     7.147395
   14  310420    1.218400e+09    1.301581e+09     6.827118
   15  852351    1.072738e+10    1.145349e+10     6.768784

[검토 